# Clinical RAG Platform — genuine Colab GPU setup
Use a GPU runtime. The notebook host may use Python 3.13; GPU packages run in an isolated **Python 3.12** environment. Run the six code cells in order.
Add `COLAB_API_KEY`, `COLAB_REGISTRATION_SECRET` and `ASPNET_BACKEND_URL` to Colab Secrets, with notebook access enabled. The backend URL is your computer's public HTTPS API origin, not localhost. Never print or paste secret values into cells.
Upload only the source ZIP made by `scripts/prepare_colab_zip.py`. Do not upload .env files, appsettings, key rings, databases, patient PDFs or evaluation inputs.
**Completion:** wait for the printed marker in each cell and for its running/stop icon to disappear. Upload completion is not installation completion. Never continue after an ERROR or cancelled installation.
This starts real Qwen3-4B-Instruct, PubMedBERT and BGE models; offline tests are not live/clinical evidence. Draft publication remains unavailable until the trusted test runner is implemented.

In [ ]:
# Cell 1: Install the tunnel binary; no model packages yet.
import os, shutil, urllib.request
if not shutil.which('cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/usr/local/bin/cloudflared')
    os.chmod('/usr/local/bin/cloudflared', 0o755)
print('TUNNEL TOOL READY — run the upload cell.')

In [ ]:
# Cell 2: Upload ONLY. Choose clinical-rag-colab-source.zip.
from google.colab import files
from pathlib import Path
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Select exactly one source ZIP.')
archive_name = next(iter(uploaded))
if not archive_name.lower().endswith('.zip'):
    raise RuntimeError('Select the source ZIP, not a PDF or notebook.')
ARCHIVE_PATH = Path(archive_name).resolve()
del uploaded
print('UPLOAD COMPLETE — archive received. Run the installation cell next.')

In [ ]:
# Cell 3: Validate/extract sources and install binary GPU packages in Python 3.12.
import hashlib, json, os, stat, subprocess, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath
SOURCE_ROOT = Path(tempfile.mkdtemp(prefix='clinical-rag-')) / 'app'
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or len(names) > 1000 or sum(i.file_size for i in archive.infolist()) > 32*1024*1024:
        raise RuntimeError('Invalid or oversized source archive.')
    manifest = json.loads(archive.read('colab-source-manifest.json'))
    expected = {f['path']: f for f in manifest['files']}
    if manifest.get('format') != 1 or manifest.get('python') != '3.12' or len(expected) != len(manifest['files']) or set(names) != set(expected) | {'colab-source-manifest.json'}:
        raise RuntimeError('Source manifest does not match the ZIP.')
    for item in archive.infolist():
        name = PurePosixPath(item.filename)
        if name.is_absolute() or '..' in name.parts or chr(92) in item.filename or ':' in item.filename or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Unsafe source archive member.')
        if item.filename in expected:
            data = archive.read(item)
            if len(data) != expected[item.filename]['bytes'] or hashlib.sha256(data).hexdigest() != expected[item.filename]['sha256']:
                raise RuntimeError('Source integrity verification failed.')
    archive.extractall(SOURCE_ROOT)
install_env = os.environ.copy()
for name in ('COLAB_API_KEY', 'COLAB_KEY', 'COLAB_REGISTRATION_SECRET', 'ASPNET_BACKEND_URL', 'COLAB_MONGODB_URI'):
    install_env.pop(name, None)
subprocess.run([sys.executable, str(SOURCE_ROOT / 'colab/bootstrap.py'), '--install', str(SOURCE_ROOT)], env=install_env, check=True)
COLAB_PYTHON = SOURCE_ROOT / '.venv-colab/bin/python'
print('INSTALL COMPLETE — run the offline-test cell next.')

In [ ]:
# Cell 4: source-only offline regressions.
# Repo-hygiene, CI-governance, clinical-eval, loadtest-script and ingestion-fixture tests require the full repo or optional tooling.
r = subprocess.run([str(COLAB_PYTHON), '-m', 'pytest', '-q', 'tests', '--ignore=tests/test_repository_hygiene.py', '--ignore=tests/test_ci_governance.py', '--ignore=tests/test_clinical_eval.py', '--ignore=tests/test_loadtest_scripts.py', '--ignore=tests/test_ingestion_fixtures.py'], cwd=SOURCE_ROOT, capture_output=True, text=True)
print(r.stdout[-9000:])
print(r.stderr[-2500:])
assert r.returncode == 0
print('OFFLINE TESTS PASSED — run the model-server cell next.')


In [ ]:
# Cell 5: Launch real models; keys stay in the child process environment, not files/output.
from google.colab import userdata
if 'clinical_rag_model_process' in globals() and clinical_rag_model_process.poll() is None:
    raise RuntimeError('This notebook already owns a running model server. Do not launch a duplicate.')
model_env = os.environ.copy()
model_env['COLAB_API_KEY'] = userdata.get('COLAB_API_KEY')
model_env['PYTHONUNBUFFERED'] = '1'
model_log = open(SOURCE_ROOT / 'model-startup.log', 'w', encoding='utf-8')
clinical_rag_model_process = subprocess.Popen([str(COLAB_PYTHON), '-m', 'colab.notebook_runner', 'serve'], cwd=SOURCE_ROOT, env=model_env, stdout=model_log, stderr=subprocess.STDOUT)
print('MODEL SERVER STARTED — loading real GPU models. Run the readiness/registration cell next.')

In [ ]:
# Cell 6: Wait for real models, start the model tunnel and register/heartbeat.
import threading
if clinical_rag_model_process.poll() is not None:
    raise RuntimeError('Model startup stopped. Check the GPU/package startup before continuing.')
if 'clinical_rag_registration_process' in globals() and clinical_rag_registration_process.poll() is None:
    raise RuntimeError('Registration is already running. Do not create another tunnel/heartbeat.')
registration_env = os.environ.copy()
for name in ('COLAB_API_KEY', 'ASPNET_BACKEND_URL', 'COLAB_REGISTRATION_SECRET'):
    registration_env[name] = userdata.get(name)
registration_env['PYTHONUNBUFFERED'] = '1'
clinical_rag_registration_process = subprocess.Popen([str(COLAB_PYTHON), '-m', 'colab.notebook_runner', 'register'], cwd=SOURCE_ROOT, env=registration_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
registration_done = threading.Event()
registration_succeeded = threading.Event()
def show_registration_status():
    for line in clinical_rag_registration_process.stdout:
        if line.startswith(('Waiting for genuine', 'COLAB REGISTERED', 'REGISTRATION FAILED')):
            print(line.rstrip(), flush=True)
        if line.startswith('COLAB REGISTERED'):
            registration_succeeded.set(); registration_done.set()
    registration_done.set()
threading.Thread(target=show_registration_status, daemon=True).start()
if not registration_done.wait(timeout=1020) or not registration_succeeded.is_set():
    if clinical_rag_registration_process.poll() is None:
        clinical_rag_registration_process.terminate()
    raise RuntimeError('Registration did not succeed. Do not start application tests until /ready returns 200.')
print('STARTUP COMPLETE — on your computer check http://localhost:5100/ready for ready: true. Keep this GPU runtime running.')